# D17 · RAG (Retrieval-Augmented Generation)

**Formación Pública — Bootcamp de Ciencia de Datos para funcionarios públicos**
Bloque avanzado · IA aplicada (opcional) · Semana 20

---

## ¿Qué vas a lograr?

En D16 viste el gran poder de un LLM... y también su gran problema: **solo sabe lo que vio en su entrenamiento**, y cuando no sabe, **inventa con seguridad**. Pregúntale por el detalle de *tu* licitación, *tu* norma interna o *tu* ficha de compra, y no tiene forma de saberlo. Para el Estado, eso es inaceptable.

**RAG** (Generación Aumentada por Recuperación) es la técnica que resuelve esto, y es hoy la forma más usada de poner LLMs a trabajar con información propia. La idea es elegante: en vez de confiar en la memoria del modelo, **primero buscamos los documentos relevantes** y luego se los **entregamos al modelo** para que responda **basándose en ellos**, citando la fuente.

Al terminar serás capaz de:

- Entender el problema que RAG resuelve (memoria limitada + alucinaciones).
- Construir las tres piezas de un RAG: **recuperar → aumentar → generar**.
- Hacer que el modelo responda **solo con tus documentos**.
- Y que **reconozca cuándo no sabe** en vez de inventar.

**Competencia de salida:** construir un sistema de preguntas y respuestas sobre documentos reales del Estado, con respuestas fundamentadas y honestas.

**Entregable:** que las **4 celdas de chequeo** muestren ✅.

> **Funciona en vivo o en caché**, igual que D16: con tu API key gratuita habla con Gemini; sin ella, usa respuestas en caché y los ejercicios se completan igual.


## Tabla de Datos Reales

En esta lección utilizaremos fragmentos reales de la **Ficha de Licitación 2239-8-LR25** de ChileCompra (Convenio Marco de productos de aseo e higiene). A continuación se muestra la base de conocimiento estructurada que nuestro RAG utilizará para responder consultas:

| ID Pasaje | Fuente | Fragmento de Texto |
| :--- | :--- | :--- |
| **p1** | Licitación 2239-8-LR25 | Convenio Marco para la adquisición de productos de aseo e higiene, realizado por la Dirección de Compras y Contratación Pública. |
| **p2** | Licitación 2239-8-LR25 | Es una licitación pública del tipo LR, igual o superior a 5.000 UTM. |
| **p3** | Licitación 2239-8-LR25 | La duración del contrato es de 36 meses. |
| **p4** | Licitación 2239-8-LR25 | La evaluación de las ofertas pondera: precio 75%, experiencia 20%, programa de integridad 3% y requisitos formales 2%. |
| **p5** | Licitación 2239-8-LR25 | Esta licitación no exige garantía de seriedad de la oferta. |
| **p6** | Licitación 2239-8-LR25 | La garantía de fiel cumplimiento del contrato es por 400.000 pesos y puede entregarse de forma electrónica según la Ley 19.799, o físicamente en la Oficina de Partes, dentro de 20 días hábiles desde la notificación de la adjudicación. |
| **p7** | Licitación 2239-8-LR25 | Las preguntas se reciben hasta el 3 de junio de 2025 y las respuestas se publican el 25 de junio de 2025. El cierre de ofertas es el 31 de julio de 2025. |
| **p8** | Licitación 2239-8-LR25 | La licitación fue adjudicada el 30 de septiembre de 2025. |

*Fuente: Dirección de Compras y Contratación Pública (ChileCompra / MercadoPúblico)*


## 1. La idea de RAG: examen con libro abierto

Imagina dos formas de rendir un examen sobre el reglamento de compras:

- **A libro cerrado:** respondes de memoria. Si no te acuerdas, **inventas**. Eso es un LLM solo.
- **A libro abierto:** primero **buscas la página correcta**, la lees, y respondes **con eso a la vista**. Eso es **RAG**.

RAG no hace al modelo "más inteligente": le pone **la página correcta enfrente**. Por eso las respuestas dejan de ser memoria difusa y pasan a estar **fundamentadas en documentos que tú controlas**.

### Las tres piezas

1. **Recuperar (retrieve):** dada una pregunta, encontrar en tu colección los **pasajes más relevantes**. Hoy lo haremos con una búsqueda por **palabras en común** (lo que aprendiste en D15). Los sistemas avanzados usan *embeddings* —vectores que capturan significado— pero la idea es la misma: traer los trozos que más se parecen a la pregunta.
2. **Aumentar (augment):** construir un **prompt** que incluya esos pasajes como contexto, junto con la pregunta y una instrucción clave: *"responde solo con esta información"*.
3. **Generar (generate):** el LLM redacta la respuesta **a partir del contexto entregado**, no de su memoria.

### Lo más importante: fundamentar y abstenerse

Dos comportamientos hacen la diferencia entre un asistente confiable y uno peligroso:

- **Fundamentar (grounding):** responder **usando los documentos**, no la imaginación del modelo.
- **Abstenerse:** si la respuesta **no está** en los documentos, decir *"no lo sé"* en vez de inventar. Suena simple, pero es exactamente lo que evita los errores más graves en un contexto público.

Hoy construirás un RAG que hace las dos cosas, sobre una **ficha de licitación real**.


## 2. Preparación: documentos, NLP y el LLM

Cargamos tres cosas:

- **`KB`** (*knowledge base*): fragmentos **reales** de la ficha de la licitación **2239-8-LR25** (un Convenio Marco de artículos de aseo e higiene). Cada fragmento es un "pasaje" con su `id`, su `fuente` y su `texto`.
- **`tokenizar`**: la función de D15 para partir texto en palabras útiles (la base de la recuperación).
- **`preguntar_llm`**: la conexión al LLM con el patrón "en vivo o caché" de D16.

Ejecuta la celda.


In [ ]:
# Instala el SDK oficial de Google (en Colab; si ya está, no hace nada)
!pip install -q google-genai

import os, re, unicodedata, urllib.request
import pandas as pd

MODELO = "gemini-2.5-flash"

# ---- Base de conocimiento: fragmentos REALES de la ficha 2239-8-LR25 ----
# Si el archivo no existe localmente (ej: en Colab), intentamos descargarlo desde GitHub
if not os.path.exists("licitacion.csv"):
    try:
        url = "https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/C2-rag/licitacion.csv"  # Reemplazar por la URL real del repositorio al publicar
        urllib.request.urlretrieve(url, "licitacion.csv")
        print("licitacion.csv descargado automáticamente.")
    except Exception:
        print("Aviso: No se pudo descargar licitacion.csv automáticamente.")

df_kb = pd.read_csv("licitacion.csv")
KB = df_kb.to_dict(orient="records")

# ---- Tokenizador (de D15) ----
STOPWORDS = {"de","y","e","o","u","para","el","la","los","las","un","una","en","con",
             "a","del","al","por","su","sus","se","que","es","son","fue"}

def normalizar(texto):
    texto = texto.lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"[^a-z0-9\s]", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()

def tokenizar(texto):
    return [t for t in normalizar(texto).split() if t not in STOPWORDS and len(t) >= 3]

# ---- LLM: patrón 'en vivo o caché' (de D16) ----
def _cargar_api_key():
    try:
        from google.colab import userdata
        k = userdata.get("GEMINI_API_KEY")
        if k:
            return k
    except Exception:
        pass
    return os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")

API_KEY = _cargar_api_key()

CACHE = {
    "rag": "Según los documentos de la licitación, el precio pondera un 75% de la evaluación de las ofertas.",
    "generico": "No tengo esa información en los documentos disponibles.",
}

def preguntar_llm(prompt, tarea="rag"):
    # En vivo si hay key; si no, respuesta en caché (offline).
    if API_KEY:
        try:
            from google import genai
            client = genai.Client(api_key=API_KEY)
            return (client.models.generate_content(model=MODELO, contents=prompt).text or "")
        except Exception as e:
            print(f"⚠️ La llamada en vivo falló ({e}). Uso la respuesta en caché.")
    return CACHE.get(tarea, CACHE["generico"])

def _sa(s):
    s = unicodedata.normalize("NFKD", str(s).lower())
    return "".join(c for c in s if not unicodedata.combining(c))

print(f"{len(KB)} pasajes en la base de conocimiento cargados desde licitacion.csv.")
print("Modo:", "EN VIVO" if API_KEY else "CACHÉ (offline). Los ejercicios funcionan igual.")


## Manejo de Errores Comunes

Al construir e interactuar con sistemas RAG en Python, es habitual encontrarse con algunos de los siguientes errores. Aquí tienes cómo entenderlos y solucionarlos:

1. **`APIKeyError` o errores de cuota de Gemini:**
   Si ejecutas en modo *EN VIVO* y ves un error relacionado con la API key, verifica que hayas guardado el secreto `GEMINI_API_KEY` en los Secrets de Google Colab (icono de llave) y que hayas activado el acceso al cuaderno. Si el error es de cuota/límite excedido, el código automáticamente caerá al modo *CACHÉ* (offline), por lo que podrás seguir completando los ejercicios sin problemas.

2. **`IndexError: list index out of range` o similares al generar:**
   Esto ocurre si tu función `recuperar` no encuentra pasajes relevantes (por ejemplo, ante una consulta no relacionada) y devuelve una lista vacía, pero luego intentas acceder a `pasajes[0]` en tu código. Asegúrate de verificar siempre si la lista de pasajes recuperados está vacía antes de procesarla.

3. **Alucinaciones o respuestas ignorando el contexto:**
   Si el LLM responde cosas que no están en la licitación (por ejemplo, diciendo que la capital de Francia es París cuando le preguntaste algo fuera de alcance), revisa la redacción de tu prompt en `armar_prompt`. Es indispensable incluir la frase restrictiva: `"Responde la pregunta usando SOLO la información de los Documentos"` y `"Si la respuesta no está en los Documentos, dilo claramente"`. Sin estas instrucciones de anclaje (*grounding*), el modelo usará su memoria preentrenada en lugar de limitarse a tus pasajes.

4. **`TypeError: unsupported operand type(s) for &`:**
   Este error surge al intentar realizar la intersección de conjuntos (`&`) si una de las dos variables no es un conjunto (`set`). En el Ejercicio 2, asegúrate de que tanto los tokens de la consulta como los tokens guardados en `INDICE` sean de tipo `set`.


## Ejercicio 1 — Indexar los documentos

Para poder **buscar rápido**, primero preparamos un **índice**: a cada pasaje le calculamos su conjunto de palabras útiles (sus *tokens*). Así, comparar una pregunta con un pasaje será comparar conjuntos de palabras.

Programa la función `indexar(kb)` que devuelva un **diccionario** `{id_del_pasaje: conjunto_de_tokens}`.

> 💡 **Pista:** usa `tokenizar(p["texto"])` y conviértelo en `set`. Una comprensión de diccionario lo resuelve:
> `{p["id"]: set(tokenizar(p["texto"])) for p in kb}`

> ⚠️ **Por qué un `set`:** para medir "palabras en común" nos da igual cuántas veces se repite una palabra; solo si está o no. Los conjuntos hacen la intersección directa y rápida.


In [ ]:
def indexar(kb):
    return {p["id"]: set(tokenizar(p["texto"])) for p in kb}

INDICE = indexar(KB)
print(INDICE["p4"])

In [ ]:
# Celda de chequeo — Ejercicio 1
def _chequeo_1():
    try:
        idx = INDICE
    except NameError:
        print("❌ No existe `INDICE`. Define `indexar` y ejecuta la celda."); return
    if not idx:
        print("❌ `INDICE` está vacío. Completa `indexar`."); return
    try:
        assert isinstance(idx, dict), "El índice debe ser un diccionario."
        assert len(idx) == len(KB), f"Debe tener {len(KB)} entradas, una por pasaje."
        assert all(isinstance(v, set) for v in idx.values()), "Cada valor debe ser un conjunto (set) de tokens."
        assert "precio" in idx["p4"] and "pondera" in idx["p4"], "El pasaje p4 debería incluir 'precio' y 'pondera'."
        print("✅ Correcto: índice construido (un conjunto de palabras por pasaje).")
    except AssertionError as e:
        print(f"❌ Casi. Pista: {e}")

_chequeo_1()

## Ejercicio 2 — Recuperar: encontrar los pasajes relevantes

Este es el corazón del *retrieval*. Dada una pregunta, queremos los pasajes que **más palabras comparten** con ella.

Programa la función `recuperar(consulta, k=2)` que:

1. Tokenice la `consulta` y la guarde como conjunto.
2. Para cada pasaje de `KB`, calcule su **puntaje** = cuántas palabras comparte (intersección con `INDICE[id]`).
3. Devuelva los **`k` pasajes con mayor puntaje**, descartando los de puntaje **0** (irrelevantes).

> 💡 **Pista:** acumula `(puntaje, pasaje)` para los de puntaje > 0, ordena de mayor a menor por el puntaje y toma los primeros `k`.

> ⚠️ **Detalle clave:** descartar el puntaje 0 es lo que después permite **abstenerse**. Si nada comparte palabras con la pregunta, no hay nada que recuperar... y eso es una señal valiosa.


In [ ]:
def recuperar(consulta, k=2):
    consulta_tokens = set(tokenizar(consulta))
    puntuados = []
    for p in KB:
        puntaje = len(consulta_tokens & INDICE[p["id"]])
        if puntaje > 0:
            puntuados.append((puntaje, p))
    puntuados.sort(key=lambda par: par[0], reverse=True)
    return [p for puntaje, p in puntuados[:k]]

for p in recuperar("¿qué porcentaje pondera el precio?"):
    print(p["id"], "→", p["texto"][:60], "...")

In [ ]:
# Celda de chequeo — Ejercicio 2
def _chequeo_2():
    try:
        f = recuperar
    except NameError:
        print("❌ Aún no defines `recuperar`."); return
    try:
        r1 = f("¿qué porcentaje pondera el precio en la evaluación?")
        assert r1, "No recuperó nada para una pregunta que sí está en los documentos."
        assert r1[0]["id"] == "p4", f"El pasaje más relevante debería ser p4 (criterios), no {r1[0]['id']}."
        r2 = f("¿cuánto es la garantía de fiel cumplimiento?")
        assert r2 and r2[0]["id"] == "p6", "Para la garantía de fiel cumplimiento, el mejor pasaje es p6."
        r3 = f("¿cuál es la capital de Francia?")
        assert r3 == [], "Una pregunta ajena a los documentos no debería recuperar nada (lista vacía)."
        print("✅ Correcto: recuperas lo relevante y descartas lo que no comparte palabras.")
        print("   «precio…» → top:", r1[0]["id"], "  ·  «garantía…» → top:", r2[0]["id"])
    except AssertionError as e:
        print(f"❌ Casi. Pista: {e}")
    except Exception as e:
        print(f"❌ La función falló: {e}. Revisa el puntaje y el orden.")

_chequeo_2()

## Ejercicio 3 — Aumentar: el prompt con contexto

Ya tenemos los pasajes relevantes. Ahora construimos el **prompt aumentado**: le entregamos al modelo esos pasajes como contexto, la pregunta, y la instrucción de oro: **responder solo con esa información**.

Programa la función `armar_prompt(consulta, pasajes)` que devuelva un texto con:

1. Una instrucción que diga que responda **SOLO** con la información de los documentos, y que si no está, lo diga.
2. Los **pasajes** recuperados (su texto), como contexto.
3. La **pregunta** del usuario.

> 💡 **Pista:** une los pasajes en un bloque de texto y arma el prompt por partes:
> ```python
> contexto = "\n".join(f"[{p['id']}] {p['texto']}" for p in pasajes)
> prompt = ("Responde usando SOLO la información de los Documentos.\n"
>           "Si no está, dilo.\n\n"
>           f"Documentos:\n{contexto}\n\n"
>           f"Pregunta: {consulta}\nRespuesta:")
> ```

> ⚠️ **El detalle que todo lo cambia:** sin la frase "responde SOLO con esto", el modelo vuelve a su memoria y puede alucinar. Esa instrucción es lo que **ancla** la respuesta a tus documentos.


In [ ]:
def armar_prompt(consulta, pasajes):
    contexto = "\n".join(f"[{p['id']}] {p['texto']}" for p in pasajes)
    prompt = ("Responde la pregunta usando SOLO la información de los Documentos.\n"
              "Si la respuesta no está en los Documentos, dilo claramente.\n\n"
              f"Documentos:\n{contexto}\n\n"
              f"Pregunta: {consulta}\nRespuesta:")
    return prompt

demo = [{"id": "p4", "fuente": "Licitación 2239-8-LR25",
         "texto": "La evaluación de las ofertas pondera: precio 75%, experiencia 20%."}]
print(armar_prompt("¿Cuánto pondera el precio?", demo))

In [ ]:
# Celda de chequeo — Ejercicio 3
def _chequeo_3():
    try:
        f = armar_prompt
    except NameError:
        print("❌ Aún no defines `armar_prompt`."); return
    demo = [{"id": "p4", "fuente": "x", "texto": "La evaluación pondera: precio 75%."}]
    try:
        prompt = f("¿Cuánto pondera el precio?", demo)
        assert prompt is not None, "La función devuelve None. Complétala."
        assert isinstance(prompt, str), "El prompt debe ser un texto."
        assert "precio 75%" in prompt, "El prompt debe incluir el TEXTO de los pasajes recuperados."
        assert "¿Cuánto pondera el precio?" in prompt, "El prompt debe incluir la pregunta del usuario."
        assert "solo" in _sa(prompt) and "documentos" in _sa(prompt), \
            "Falta la instrucción de responder SOLO con los Documentos."
        print("✅ Correcto: prompt aumentado con contexto + pregunta + instrucción de fundamentar.")
    except AssertionError as e:
        print(f"❌ Casi. Pista: {e}")

_chequeo_3()

## Ejercicio 4 — Generar (y saber decir "no lo sé")

Última pieza: unir todo. Recuperamos, aumentamos y **generamos** la respuesta con el LLM. Pero con una regla de oro: si la recuperación **no trae nada**, **no llamamos al modelo** y respondemos honestamente que no está en los documentos. Así garantizamos la **abstención** incluso si el modelo tuviera ganas de inventar.

Programa la función `responder(consulta)` que:

1. Llame a `recuperar(consulta, k=2)`.
2. Si **no hay pasajes**, devuelva exactamente: `"No encontré esa información en los documentos disponibles."`
3. Si hay pasajes, arme el prompt con `armar_prompt` y devuelva `preguntar_llm(prompt, tarea="rag")`.

> 💡 **Pista:** `pasajes = recuperar(consulta, k=2)`; si `not pasajes:` devuelves el mensaje de abstención; si no, generas.

> ⚠️ **Esto es lo que hace confiable a un RAG público:** ante una pregunta cuya respuesta no está en tus documentos, **prefiere callar a inventar**.


In [ ]:
def responder(consulta):
    pasajes = recuperar(consulta, k=2)
    if not pasajes:
        return "No encontré esa información en los documentos disponibles."
    prompt = armar_prompt(consulta, pasajes)
    return preguntar_llm(prompt, tarea="rag")

print("P1:", responder("¿Qué porcentaje pondera el precio en la evaluación?"))
print("P2:", responder("¿Cuál es la capital de Francia?"))

In [ ]:
# Celda de chequeo — Ejercicio 4
def _chequeo_4():
    try:
        f = responder
    except NameError:
        print("❌ Aún no defines `responder`."); return
    try:
        fuera = f("¿Cuál es la capital de Francia?")
        assert fuera is not None, "La función devuelve None. Complétala."
        assert "no encontr" in _sa(fuera), \
            "Ante una pregunta ajena a los documentos, debe ABSTENERSE (no inventar)."
        dentro = f("¿Qué porcentaje pondera el precio en la evaluación?")
        assert isinstance(dentro, str) and len(dentro.strip()) > 0, "La respuesta fundamentada llegó vacía."
        assert "no encontr" not in _sa(dentro), \
            "Esta pregunta SÍ está en los documentos: no debería abstenerse."
        print("✅ Correcto: RAG completo. Fundamenta cuando sabe y se abstiene cuando no.")
        print("   Dentro de los docs →", dentro.strip()[:90], "...")
        print("   Fuera de los docs  →", fuera)
    except AssertionError as e:
        print(f"❌ Casi. Pista: {e}")
    except Exception as e:
        print(f"❌ La función falló: {e}.")

_chequeo_4()

## Cierre — Lo que te llevas

- **RAG = recuperar → aumentar → generar.** No hace al modelo más listo: le pone **la página correcta enfrente**.
- La **recuperación** puede empezar simple (palabras en común, como en D15); los sistemas avanzados usan *embeddings* que capturan significado, pero el flujo es el mismo.
- El prompt **aumentado** con la instrucción "responde SOLO con esto" es lo que **ancla** la respuesta a tus documentos.
- Y lo más valioso en el Estado: **abstenerse**. Un asistente que dice "no lo sé" cuando no está en los documentos es infinitamente más confiable que uno que inventa.

### Para llevar al trabajo
Cualquier conjunto de documentos propios —un reglamento interno, un set de bases tipo, las preguntas frecuentes de tu unidad— puede convertirse en un asistente de preguntas y respuestas con esta misma receta. Y como cita la fuente, **siempre puedes verificar**.

### Hacia dónde sigue
En **D18** darás el último paso: encadenar estas capacidades (clasificar, extraer, recuperar, responder) en un **agente** que decide qué herramienta usar para resolver una tarea de principio a fin.

---
*Formación Pública · Contenido bajo licencia CC BY 4.0. Documento base: ficha real de licitación (ChileCompra / MercadoPúblico). LLM: Google Gemini (capa gratuita).*
